# Smart Nyuki — Hive State from Sound + Sensors (Smart Bee Colony Monitor dataset)
Beyond *queen present / absent*: this dataset (Anna Yang, Kaggle) labels **queen status in four states — original queen, queenless, rejected new queen, accepted new queen** — and pairs each 60 s recording with **temperature, humidity and pressure**. That is the same kind of data Smart Nyuki's DHT11 already collects.

What you can predict with it:
- **Queen acceptance** — when it is safe to release a caged new queen
- **Queenright vs queenless** from sensors alone (no mic), audio alone, or both fused

The key lesson in this notebook: compare a **random split** with **leave-one-hive-out**. Published work (BeeTogether, Sensors 2024) found queen-classification frequency depends mostly on *which colony* is recorded — so random-split scores overstate how well a model works on a new hive.

In [ ]:
!pip -q install librosa kagglehub joblib

In [ ]:
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import librosa
from joblib import Parallel, delayed
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import f1_score, accuracy_score, ConfusionMatrixDisplay
SEED = 42
SR, CLIP_SECONDS = 8000, 30

## 1. Download
Dataset: **Smart Bee Colony Monitor: Clips of Beehive Sounds** (DOI 10.34740/KAGGLE/DSV/4451415). If the slug below fails, search Kaggle for that title and paste the `owner/dataset` part of its URL.

In [ ]:
import kagglehub
DATA_DIR = kagglehub.dataset_download('annajyang/beehive-sounds')
csvs = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.csv'), recursive=True), key=os.path.getsize)
print('CSV files:', [os.path.relpath(c, DATA_DIR) for c in csvs])
meta = pd.read_csv(csvs[-1])                 # the largest CSV holds the per-recording metadata
print(meta.shape); print(meta.columns.tolist()); meta.head()

## 2. Pick the columns
Auto-detected below — **check the printout** and override any that are wrong. Then fill `LABEL_NAMES` from the dataset's description page so the plots show names instead of codes.

In [ ]:
lower = {c.lower().strip(): c for c in meta.columns}
def find(*keys):
    return next((c for lc, c in lower.items() if all(k in lc for k in keys)), None)

HIVE_COL = find('hive', 'number') or find('hive', 'id')
TARGET_COL = find('queen', 'status') or find('queen', 'presence')
FILE_COL = find('file')
EXCLUDE = ('queen', 'target', 'file', 'date', 'time', 'device', 'hive number', 'hive id', 'lat', 'lon')
ENV_COLS = [c for c in meta.select_dtypes('number').columns if not any(k in c.lower() for k in EXCLUDE)]
LABEL_NAMES = {}      # e.g. {0: 'original queen', 1: 'queenless', ...} — copy from the dataset page

print('hive  :', HIVE_COL); print('target:', TARGET_COL); print('file  :', FILE_COL); print('env   :', ENV_COLS)
assert HIVE_COL and TARGET_COL, 'Set HIVE_COL / TARGET_COL by hand from the column list above'
meta = meta.dropna(subset=[TARGET_COL]).reset_index(drop=True)
print(pd.crosstab(meta[HIVE_COL], meta[TARGET_COL]))

The crosstab matters: if a state only occurs in one hive, *no* model can be fairly tested on it with leave-one-hive-out — say so in your report rather than hiding it.

## 3. Audio features (MFCC + log-Mel summary per recording)
We read the first 30 s of each 60 s clip at 8 kHz to keep it fast; set `CLIP_SECONDS = None` for the full clip.

In [ ]:
AUDIO_EXT = ('.wav', '.mp3', '.flac')
audio_index = {os.path.splitext(os.path.basename(p))[0]: p
               for p in glob.glob(os.path.join(DATA_DIR, '**', '*'), recursive=True) if p.lower().endswith(AUDIO_EXT)}
print(len(audio_index), 'audio files found')

def audio_feats(path):
    try:
        y, _ = librosa.load(path, sr=SR, mono=True, duration=CLIP_SECONDS)
    except Exception:
        return None
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_fft=512, hop_length=256, n_mels=40, fmin=50, fmax=4000))
    mfcc = librosa.feature.mfcc(S=mel, n_mfcc=20)
    return np.concatenate([mfcc.mean(1), mfcc.std(1), mel.mean(1)])

HAS_AUDIO = FILE_COL is not None and len(audio_index) > 0
if HAS_AUDIO:
    keys = meta[FILE_COL].astype(str).map(lambda s: os.path.splitext(os.path.basename(s))[0])
    paths = keys.map(audio_index)
    print('metadata rows matched to audio:', paths.notna().mean().round(3))
    feats = Parallel(n_jobs=-1, verbose=5)(delayed(audio_feats)(p) if isinstance(p, str) else delayed(lambda: None)() for p in paths)
    ok = np.array([f is not None for f in feats])
    meta = meta[ok].reset_index(drop=True)
    A = np.stack([f for f in feats if f is not None])
    AUDIO_COLS = [f'mfcc{i}_mean' for i in range(20)] + [f'mfcc{i}_std' for i in range(20)] + [f'mel{i}' for i in range(40)]
    meta = pd.concat([meta, pd.DataFrame(A, columns=AUDIO_COLS)], axis=1)
else:
    AUDIO_COLS = []
    print('No audio matched — running the sensor-only experiments')

## 4. Experiments: sensors vs audio vs fusion, random split vs unseen hive

In [ ]:
y = meta[TARGET_COL].values
groups = meta[HIVE_COL].values

def model():
    return make_pipeline(SimpleImputer(strategy='median'),
                         RandomForestClassifier(n_estimators=400, class_weight='balanced', n_jobs=-1, random_state=SEED))

feature_sets = {'sensors only (DHT-like)': ENV_COLS}
if AUDIO_COLS:
    feature_sets['audio only'] = AUDIO_COLS
    feature_sets['fusion'] = ENV_COLS + AUDIO_COLS

rows, preds = [], {}
for name, cols in feature_sets.items():
    X = meta[cols].values
    for split, cv, g in [('random 5-fold', StratifiedKFold(5, shuffle=True, random_state=SEED), None),
                         ('leave-one-hive-out', LeaveOneGroupOut(), groups)]:
        p = cross_val_predict(model(), X, y, cv=cv, groups=g)
        preds[(name, split)] = p
        rows.append(dict(features=name, split=split, accuracy=accuracy_score(y, p), macro_f1=f1_score(y, p, average='macro')))
results = pd.DataFrame(rows).pivot(index='features', columns='split', values=['accuracy', 'macro_f1']).round(3)
results

In [ ]:
best_name = 'fusion' if 'fusion' in feature_sets else 'sensors only (DHT-like)'
labels = sorted(np.unique(y))
names = [LABEL_NAMES.get(l, str(l)) for l in labels]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, split in zip(axes, ['random 5-fold', 'leave-one-hive-out']):
    ConfusionMatrixDisplay.from_predictions(y, preds[(best_name, split)], labels=labels, display_labels=names,
                                            ax=ax, cmap='YlOrBr', colorbar=False, xticks_rotation=30)
    ax.set_title(f'{best_name} — {split}')
plt.tight_layout(); plt.show()

In [ ]:
cols = feature_sets[best_name]
final = model().fit(meta[cols].values, y)
imp = pd.Series(final[-1].feature_importances_, cols).sort_values().tail(15)
imp.plot.barh(figsize=(7, 6), title=f'Top features — {best_name}'); plt.show()

## 5. What this means for Smart Nyuki
- If **sensors only** scores close to fusion on unseen hives, the DHT11 + load cell stack can flag queen problems **without a microphone** — cheapest path.
- If **audio** adds a lot, it justifies the I²S mic and the TinyML model from the queen-audio notebook.
- The **gap between the two split columns** is how much the model is memorising individual hives. Report the leave-one-hive-out number — it is the honest one.
- With only a few hives here, the next real gain comes from **recording Smart Nyuki's own hives** and labelling them at each inspection.